# WSJ 2027 - Gruppindelning Rundresa

Assign confirmed rundresa deltagare into groups of exactly 36.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — participants should live close to each other (Hilbert curve sort)
3. **Friend wish** — at least ONE of friend_1/friend_2 in same group (soft goal)
4. **Max 8 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex should be as evenly spread as possible

In [1]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

# Fetch and parse all participants
raw_data = u.fetch_participants()
df, skipped = u.build_participant_dataframe(raw_data)

Fetched 1239 participants
Confirmed: 1180, Unconfirmed: 26, Cancelled: 33
Total confirmed participants: 1180
Skipped: 26 unconfirmed, 0 wrong age/no DOB

By category:
category
deltagare    989
ist          169
cmt           22

By travel type:
travel
rundresa      837
direktresa    202
egen_resa     119
other          22

Friend wishes:
  With friend 1 member no: 647
  With friend 2 member no: 366
  With friend 1 name (text): 57
  With friend 2 name (text): 38


In [2]:
GROUP_SIZE = 36

# Filter to rundresa deltagare only
df_rundresa = df[df['travel'] == 'rundresa'].copy().reset_index(drop=True)
print(f"=== Rundresa deltagare ===")
print(f"Participants: {len(df_rundresa)}")

# Assign coordinates and Hilbert sort
u.assign_coordinates(df_rundresa)
df_sorted = u.add_hilbert_index(df_rundresa)

n_full = len(df_sorted) // GROUP_SIZE
remainder = len(df_sorted) % GROUP_SIZE
total_groups = n_full + (1 if remainder > 0 else 0)
print(f"\nGroups: {n_full} x {GROUP_SIZE} + 1 x {remainder} = {total_groups} total")
print(f"\nBy region:")
print(df_sorted['region'].value_counts().to_string())
print(f"\nBy age:")
print(df_sorted['age'].value_counts().sort_index().to_string())
print(f"\nBy sex:")
print(df_sorted['sex'].map(u.SEX_MAP).value_counts().to_string())

=== Rundresa deltagare ===
Participants: 837
With coordinates: 835
Without coordinates (Sweden centroid): 2

Groups: 23 x 36 + 1 x 9 = 24 total

By region:
region
Region Stockholm    260
Region Södra        209
Region Västra       162
Region Norr/Mitt     91
Region Östra         59
                      2

By age:
age
14    196
15    229
16    186
17    176
18     22
19     14
20      2
21      6
22      2
23      2
25      2

By sex:
sex
Kvinna    433
Man       398
Annat       4
Okänt       2


In [3]:
# Resolve text-only friend wishes via fuzzy name matching
u.resolve_friend_wishes(df_sorted, df)
friend_wishes = u.build_friend_graph(df_sorted)

=== Text Friend Name Matching ===
Text-only wishes: 60
Matched & applied: 24
Generic wishes (not a person): 3
Unresolved (friend not in project): 33

Matched:
  Albin Åkesson -> Malte Lindsjö (Jonstorps Kustscoutkår) [rundresa] via fuzzy(0.78)
  Alex Liljerot Priftis -> Vera Tollwé (Årsta Scoutkår) [rundresa] via fuzzy(0.86)
  Alma Stössel Weinryb -> Lotten Hellman (Södermalms Scoutkår) [rundresa] via exact
  Alma Stössel Weinryb -> Alfons Ekelin Sintorn (Södermalms Scoutkår) [rundresa] via fuzzy(0.76)+kar
  Charlie Lindberg -> Axel Lindroth (Saltsjö-Boo Scoutkår) [rundresa] via fuzzy(0.96)
  Elsie Stenfeldt -> Hugo Johannessen (Tuve Scoutkår) [rundresa] via exact
  Emilio Larsen -> Emmy Nilsson (Glimåkra Scoutkår) [rundresa] via fuzzy(0.77)
  Flora Wiklander -> Oscar Löf (Scoutkåren Gustaf Vasa-Bredäng) [direktresa] via exact
  Ida Jönsson -> Vendela Gustafsson (Huddinge Scoutkår) [rundresa] via fuzzy(0.76)
  Linnea Barath -> Erik Hartwig (Staffanstorps Scoutkår Torparna) [rundresa] v

In [4]:
# Run the full group assignment algorithm
df_sorted = u.assign_groups(df_sorted, GROUP_SIZE, friend_wishes)
total_groups = df_sorted['group'].nunique()

Participants: 837
Groups: 23 x 36 + 1 x 9 = 24 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 383/446
  Kar violations: 79
  Avg geo spread: 0.6835

=== Phase 2: Fix friend wishes ===
  Swaps: 44
  Friend satisfaction: 441/446
  Kar violations: 70
  Avg geo spread: 1.1193

=== Phase 3: Fix kar violations (geo-aware) ===
  Swaps: 64
  Kar violations: 0
  Friend satisfaction: 390/446
  Avg geo spread: 1.1183

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 33
  Friend satisfaction: 444/446
  Kar violations: 0
  Avg geo spread: 1.1659

=== Phase 4: Diversity optimization (geo-penalized) ===
  Swaps: 399
  Diversity score: 74.89 -> 76.10
  Avg geo spread: 1.1659 -> 1.4554

=== FINAL RESULTS ===
Groups: 23 x 36 + 1 x 9
Friend satisfaction: 444/446 (100%)
Kar violations: 0
Total swaps: 540
Diversity: 76.10
Avg geo spread: 1.4554


In [5]:
# Per-group quality metrics
group_of = df_sorted['group'].values
u.print_group_metrics(df_sorted, group_of, total_groups)

Grupp Storlek  Van% MaxKar     M/K/A  14  15  16  17  AvstandKm Karer
--------------------------------------------------------------------------------
    1      36 24/24      8   13/23/0  10  11   8   5        22    11
    2      36 26/26      8   18/18/0  13   9   8   4        31    15
    3      36 22/22      8   13/22/0   7  10   6   9        17    18
    4      36 18/18      5   17/19/0  14  12   3   6        37    19
    5      36 14/14      7   17/19/0   9   6   8  12       112    19
    6      36 22/22      6   16/20/0   9   8   8   9        71    16
    7      36 27/27      8   13/23/0   8   9   9  10        67    13
    8      36 18/18      6   22/14/0   8  13   9   5        59    19
    9      36 22/22      8   20/16/0   7   9   9   9        48    14
   10      36 18/18      5    8/28/0   7   8  10   8        44    21
   11      36 15/15      6   17/18/0   7   7   9  11       110    19
   12      36 19/19      5   19/16/1   6  10   8  11       130    21
   13      36 20/20  

In [6]:
# Export CSV + JSON
OUTPUT_DIR = '/config/notebooks/wsj27'
csv_path, json_path = u.export_results(df_sorted, group_of, total_groups, OUTPUT_DIR, prefix='wsj27_rundresa')

Saved 837 participants to /config/notebooks/wsj27/wsj27_rundresa_grupper.csv
Saved group summary to /config/notebooks/wsj27/wsj27_rundresa_grupper.json

CSV preview (first 10 rows):
 group                  name member_no  age    sex               kar                       district           region friend_1 friend_2       lat       lng
     1          Erik Sjölund   3385355   14    Man Bunkeflo Scoutkår     Södra Skånes Scoutdistrikt     Region Södra                   55.557482 12.963234
     1        Tuva Magnusson   3311571   18 Kvinna Bunkeflo Scoutkår     Södra Skånes Scoutdistrikt     Region Södra                   55.557482 12.963234
     1            Alva Dunér   3328940   15 Kvinna    Dalby Scoutkår     Södra Skånes Scoutdistrikt     Region Södra  3460657  3372547 55.664664 13.346159
     1             Anouk Asp   3351398   15 Kvinna    Dalby Scoutkår     Södra Skånes Scoutdistrikt     Region Södra  3355911  3386859 55.664664 13.346159
     1 Hera Hjerpe Johansson   3460657   14

In [7]:
# Generate interactive map
map_path = f'{OUTPUT_DIR}/wsj_rundresa_karta.html'
u.generate_group_map_html(df_sorted, total_groups, map_path, title='WSJ 2027 Rundresa',
                          friend_wishes=friend_wishes, show_group_arcs=True)
print(f"\nAll outputs:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Map:  {map_path}")

Friend arcs: 407 (373 satisfied, 34 unsatisfied)
Group arcs: 14526 connections across 24 groups
Saved group map: /config/notebooks/wsj27/wsj_rundresa_karta.html

All outputs:
  CSV:  /config/notebooks/wsj27/wsj27_rundresa_grupper.csv
  JSON: /config/notebooks/wsj27/wsj27_rundresa_grupper.json
  Map:  /config/notebooks/wsj27/wsj_rundresa_karta.html
